# hello.cptdaz.org (Front Door endpoint cptmtls-hello)


Public endpoint without mTLS. `/hello.txt` comes from a private storage account (Private Link), `/index.html` from the Node.js VM (Private Link Service). The page references `style.css` on `media.cptdaz.org`.

Tools: bash, az, http (httpie), jq, azcopy.

## Design notes

* **Domain and certificate:** `hello.cptdaz.org` is a CNAME to the endpoint's generated `azurefd.net` hostname. The hostname is pre-computed with `scripts/get-afd-hostname.sh`. Front Door issues a managed certificate after the `_dnsauth` TXT record validates the domain.
* **`/hello.txt`:** The storage account is private and reached through a Front Door private endpoint. It carries the exclusion tag `SecurityControl=Ignore`, so the tenant policies do not reset anonymous blob access. Anonymous read applies only behind the private endpoint.
* **`/index.html`:** Served by a Node.js server on the VM (cloud-init, systemd, built-in `http` module only) behind an internal load balancer and a Private Link Service. The page references `style.css` on `media.cptdaz.org`.
* **Approvals:** Front Door creates the private endpoint connections as `Pending`. `scripts/approve-pe.sh` (called by the workflow) approves them on the storage account and the Private Link Service.
* **Seeding:** `hello.txt` is written by the VM through its managed identity (Bicep `runCommand`), because shared key access is disabled by policy.


In [ ]:
export SUB=09808f31-065f-4231-914d-776c2d6bbe34
export RG=rg-cptmtls
export DNS_SUB=ff0bb075-6c44-44ee-bb64-d46ce828c62f
export DNS_RG=domains
export DNS_ZONE=cptdaz.org
export PROFILE=afd-cptmtls-public
export ENDPOINT=cptmtls-hello
export DOMAIN=hello-cptdaz-org
export HOST=hello.cptdaz.org
export MEDIA_HOST=media.cptdaz.org
export STORAGE=stcptmtlshelloizhwq
export VM=vm-cptmtls
export PLS=pls-cptmtls
export LAW=law-cptmtls
export PAGER=cat
export AZCOPY_AUTO_LOGIN_TYPE=AZCLI
az account set --subscription $SUB
export VM_IP=$(az vm show -d -g $RG -n $VM --query privateIps -o tsv)
export LAW_ID=$(az monitor log-analytics workspace show -g $RG -n $LAW --query customerId -o tsv)
env | grep -E '^(SUB|RG|DNS_SUB|DNS_RG|DNS_ZONE|PROFILE|ENDPOINT|DOMAIN|HOST|MEDIA_HOST|STORAGE|VM|PLS|LAW|VM_IP|LAW_ID)=' | sort

## Azure account

In [ ]:
az account show --query "{name:name,id:id,user:user.name}" -o table

## Front Door configuration

In [ ]:
az afd profile show -g $RG --profile-name $PROFILE --query "{name:name,sku:sku.name,state:provisioningState}" -o table

In [ ]:
az afd endpoint show -g $RG --profile-name $PROFILE --endpoint-name $ENDPOINT --query "{host:hostName,enabled:enabledState,deployment:deploymentStatus}" -o table

In [ ]:
az afd custom-domain show -g $RG --profile-name $PROFILE --custom-domain-name $DOMAIN --query "{host:hostName,validation:domainValidationState,provisioning:provisioningState,deployment:deploymentStatus,cert:tlsSettings.certificateType}" -o table

In [ ]:
az afd route list -g $RG --profile-name $PROFILE --endpoint-name $ENDPOINT --query "[].{route:name,patterns:join(',',patternsToMatch),forwarding:forwardingProtocol,deployment:deploymentStatus}" -o table

In [ ]:
az afd origin list -g $RG --profile-name $PROFILE --origin-group-name og-hello-storage -o json | jq '.[] | {name, hostName, enabledState, privateLinkTarget: (.sharedPrivateLinkResource.privateLink.id | split("/")[-1])}'

In [ ]:
az afd origin list -g $RG --profile-name $PROFILE --origin-group-name og-hello-vm -o json | jq '.[] | {name, hostName, enabledState, privateLinkTarget: (.sharedPrivateLinkResource.privateLink.id | split("/")[-1])}'

## DNS records in the Azure DNS zone

In [ ]:
az network dns record-set cname show --subscription $DNS_SUB -g $DNS_RG -z $DNS_ZONE -n hello --query "{fqdn:fqdn,target:CNAMERecord.cname}" -o table

In [ ]:
az network dns record-set txt show --subscription $DNS_SUB -g $DNS_RG -z $DNS_ZONE -n _dnsauth.hello --query "{fqdn:fqdn,value:TXTRecords[0].value[0]}" -o table

## Private Link connections created by Front Door

In [ ]:
az network private-endpoint-connection list --id $(az storage account show -g $RG -n $STORAGE --query id -o tsv) --query "[].{name:name,status:properties.privateLinkServiceConnectionState.status,description:properties.privateLinkServiceConnectionState.description}" -o table

In [ ]:
az network private-endpoint-connection list --id $(az network private-link-service show -g $RG -n $PLS --query id -o tsv) --query "[].{name:name,status:properties.privateLinkServiceConnectionState.status,description:properties.privateLinkServiceConnectionState.description}" -o table

## Test through Front Door with httpie

In [ ]:
http --headers GET https://$HOST/hello.txt

In [ ]:
http --body GET https://$HOST/hello.txt

In [ ]:
http --body GET https://$HOST/index.html

In [ ]:
http --body GET https://$HOST/index.html | grep stylesheet

In [ ]:
http --headers GET https://$HOST/missing.txt

In [ ]:
http --headers GET http://$HOST/hello.txt

## Storage origin: az CLI and azcopy over the private endpoint

This VM reaches the storage account through the VNet peering and the linked private DNS zone.

In [ ]:
getent hosts $STORAGE.blob.core.windows.net

In [ ]:
az storage blob list --account-name $STORAGE --container-name '$root' --auth-mode login -o json | jq '.[] | {name, size: .properties.contentLength, type: .properties.contentSettings.contentType}'

In [ ]:
azcopy list "https://$STORAGE.blob.core.windows.net/\$root"

In [ ]:
azcopy copy "https://$STORAGE.blob.core.windows.net/\$root/hello.txt" /tmp/hello-azcopy.txt

In [ ]:
cat /tmp/hello-azcopy.txt

In [ ]:
http --body GET https://$HOST/hello.txt | diff - /tmp/hello-azcopy.txt && echo identical

## VM origin: Node.js server

In [ ]:
az vm show -d -g $RG -n $VM --query "{name:name,power:powerState,ip:privateIps}" -o table

In [ ]:
http --headers GET http://$VM_IP:8080/health

In [ ]:
http --body GET http://10.0.0.10/health

In [ ]:
az vm run-command invoke -g $RG -n $VM --command-id RunShellScript --scripts "systemctl is-active hello-node; node --version" --query "value[0].message" -o tsv 2>/dev/null

## Front Door access logs in Log Analytics

In [ ]:
az monitor log-analytics query -w $LAW_ID --analytics-query "AzureDiagnostics | where ResourceProvider == 'MICROSOFT.CDN' | where ResourceId contains toupper('$PROFILE') | summarize events=count() by Category" -o json | jq -c '.[]'

In [ ]:
az monitor log-analytics query -w $LAW_ID --analytics-query "AzureDiagnostics | where ResourceProvider == 'MICROSOFT.CDN' | where ResourceId contains toupper('$PROFILE') | top 5 by TimeGenerated desc | project TimeGenerated, Category, ResourceId" -o json | jq -c '.[]'